# Создание и заполнение данных БД Postgre

In [22]:
%pip install python-dotenv psycopg2-binary


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Users/glebzilenkov/big_data_course_2026/task_4_DBeaver_Jupiter/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
import json
import psycopg2
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [36]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path, override=True)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT")
secret_hash = os.getenv("SECRET_HASH")

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=zhilenkov, DB=my_db_zhilenkov, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [46]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [55]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [56]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [ ]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры.

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки.

In [53]:
conn.rollback()

In [72]:
cursor.execute("SELECT * FROM user_logs LIMIT 10;")
rows = cursor.fetchall()
for row in rows:
    print(row)

(71904, 16316, 6, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 7, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 8, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 9, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 10, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 11, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 12, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 13, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316, 14, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 42, 2, '2', '1', 6, 4, '25.06.2022')
(71904, 16316,

## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [73]:
cursor.execute("select courseid, count(distinct depart) from user_logs ul group by ul.courseid;")
rows = cursor.fetchall()
for row in rows:
    print(row)

(71262, 1)
(71269, 1)
(71275, 1)
(71279, 1)
(71301, 1)
(71302, 1)
(71323, 1)
(71324, 1)
(71326, 1)
(71335, 1)
(71344, 1)
(71349, 1)
(71352, 1)
(71354, 1)
(71357, 1)
(71384, 1)
(71402, 1)
(71409, 1)
(71411, 1)
(71415, 1)
(71461, 1)
(71467, 1)
(71472, 1)
(71495, 3)
(71496, 1)
(71508, 2)
(71523, 1)
(71535, 1)
(71541, 2)
(71545, 1)
(71547, 2)
(71549, 2)
(71559, 1)
(71571, 3)
(71596, 1)
(71615, 1)
(71619, 1)
(71632, 2)
(71635, 1)
(71646, 1)
(71675, 1)
(71697, 1)
(71702, 1)
(71708, 1)
(71709, 1)
(71710, 1)
(71716, 1)
(71717, 1)
(71720, 1)
(71730, 1)
(71732, 1)
(71733, 1)
(71736, 2)
(71740, 1)
(71746, 1)
(71750, 1)
(71758, 1)
(71776, 1)
(71792, 1)
(71797, 1)
(71801, 1)
(71806, 1)
(71816, 1)
(71820, 1)
(71829, 1)
(71831, 1)
(71852, 2)
(71857, 2)
(71871, 1)
(71880, 1)
(71882, 1)
(71883, 1)
(71884, 2)
(71885, 1)
(71887, 1)
(71892, 3)
(71895, 1)
(71902, 1)
(71903, 1)
(71904, 2)
(71905, 1)
(71906, 1)
(71909, 1)
(71917, 1)
(71925, 1)
(71930, 1)
(71934, 1)
(71963, 1)
(71972, 1)
(71976, 1)
(71978, 1)

##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [63]:
cursor.execute("""
select
	name,
	count(ul.courseid) cnt
from departments d
join user_logs ul
on ul.depart = d.id
group by name
order by cnt desc
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

('МиХТ', 25296)
('ДиСО', 25176)
('ЛиП', 19008)
('РМПИ', 17856)
('ГМДиОПИ', 16968)
('ТОМ', 16704)
('БИиИТ', 16176)
('ПОиД', 14760)
('АЭПиМ', 14232)
('ЛиУТС', 13440)
('ПиСЗ', 13248)
('ГМУиУП', 12144)
('ГМиТТК', 11952)
('Эконом.', 11928)
('МиТОДиМ', 11760)
('ВТиП', 11232)
('ИиИБ', 10608)
('ПиЭММО', 9864)
('ЛПиМ', 9768)
('Психол.', 9552)
('CC', 9504)
('ЭиМЭ', 9192)
('РЯОЯиМК', 8736)
('ТССА', 7872)
('ВИ', 7848)
('Менеджм.', 7848)
('ЯиЛ', 7488)
('ЭПП', 7368)
('АиИИ', 6816)
('Химии', 6000)
('АСУ', 5640)
('ПМиИ', 5544)
('ХОМ', 5280)
('Дизайна', 5088)
('ТиЭС', 5016)
('УиИС', 4200)
('СРиППО', 3864)
('ПЭиБЖД', 3840)
('Физики', 3480)
('ЦДОМ', 984)
('Физкульт.', 504)
('ИТМ', 504)
('УСиБА', 240)


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [64]:
cursor.execute("""
select
	ul.courseid,
	count(distinct ul.depart) as dep_cnt,
	STRING_AGG(DISTINCT d.name, ', ' ORDER BY d.name) AS dep_names
from user_logs ul
join departments d
on d.id = ul.depart
group by courseid
order by dep_cnt desc
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(78057, 25, 'АСУ, АЭПиМ, ВИ, ВТиП, ГМДиОПИ, ГМиТТК, ГМУиУП, ДиСО, ИиИБ, ЛиП, Менеджм., МиТОДиМ, МиХТ, ПиЭММО, ПОиД, Психол., ПЭиБЖД, РМПИ, РЯОЯиМК, ТиЭС, ТОМ, ХОМ, ЭиМЭ, Эконом., ЯиЛ')
(72457, 4, 'ГМДиОПИ, ЛПиМ, МиХТ, ТОМ')
(75833, 4, 'ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ')
(72380, 4, 'ГМДиОПИ, ЛПиМ, МиХТ, ТОМ')
(72402, 4, 'ГМДиОПИ, ЛПиМ, МиХТ, ТОМ')
(79426, 4, 'ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ')
(72359, 3, 'ЛПиМ, МиХТ, ТОМ')
(72885, 3, 'Дизайна, ТОМ, ХОМ')
(87396, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(72392, 3, 'ЛПиМ, МиХТ, ТОМ')
(71892, 3, 'АЭПиМ, ТиЭС, ЭПП')
(72447, 3, 'ЛПиМ, МиХТ, ТОМ')
(83853, 3, 'ЛПиМ, МиХТ, ТОМ')
(72416, 3, 'ЛПиМ, МиХТ, ТОМ')
(75834, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(75839, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(88415, 3, 'ГМДиОПИ, МиХТ, ТОМ')
(71495, 3, 'ГМиТТК, ПиСЗ, УиИС')
(84834, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(75849, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(75810, 3, 'ГМДиОПИ, ГМиТТК, РМПИ')
(72460, 3, 'ЛПиМ, МиХТ, ТОМ')
(71571, 3, 'ЛиУТС, ПиСЗ, УиИС')
(72865, 2, 'Дизайна, ТОМ')
(76293, 2, 'ПиСЗ, Эконом.')


## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [65]:
cursor.execute("""
select
	ul.namer_level,
	count(ul.userid)
from user_logs ul
group by namer_level
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(2, 44184)
(3, 70536)
(4, 130104)
(5, 169704)


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [66]:
cursor.execute("""
select
	ul.userid,
	count(*) cnt
from user_logs ul
group by ul.userid
order by cnt desc
limit 1
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(24380, 120)


Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [68]:
cursor.execute("""
select
	ul.num_week,
	avg(s_all)
from user_logs ul
group by num_week
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(6, Decimal('13.7966072255673923'))
(7, Decimal('9.6161417322834646'))
(8, Decimal('8.0285433070866142'))
(9, Decimal('9.3932955071792497'))
(10, Decimal('8.2085456229735989'))
(11, Decimal('10.0220009263547939'))
(12, Decimal('9.3817160722556739'))
(13, Decimal('10.0140111162575266'))
(14, Decimal('9.8601783232978231'))
(15, Decimal('10.3536938397406207'))
(16, Decimal('10.2854330708661417'))
(17, Decimal('10.5158059286706809'))
(18, Decimal('9.6713756368689208'))
(19, Decimal('11.1134784622510421'))
(20, Decimal('14.4471977767484947'))
(21, Decimal('18.5038212135247800'))
(22, Decimal('22.4873205187586846'))
(23, Decimal('22.2615215377489579'))
(24, Decimal('23.0121005094951366'))
(25, Decimal('18.2163617415470125'))
(26, Decimal('8.6029411764705882'))
(27, Decimal('1.2529527559055118'))
(28, Decimal('0.08997220935618341825'))
(29, Decimal('0.05471283001389532191'))


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников.

In [69]:
cursor.execute("""
select
	d.name,
	count(distinct ul.userid) cnt
from departments d
join user_logs ul
on ul.depart = d.id
where ul.namer_level = 5
group by d."name"
order by cnt desc
limit 1;
               """)
rows = cursor.fetchall()
for row in rows:
    print(row)

('ДиСО', 310)


In [70]:
cursor.execute("""
select
	d.name,
	count(distinct ul.userid) cnt
from departments d
join user_logs ul
on ul.depart = d.id
where ul.namer_level = 2
group by d."name"
order by cnt desc
limit 1;
               """)
rows = cursor.fetchall()
for row in rows:
    print(row)

('Эконом.', 72)


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [71]:
cursor.execute("""
WITH week_activity AS (
    SELECT
        ul.num_week AS week_number,
        AVG(ul.s_all) AS avg_s_all_total
    FROM user_logs ul
    GROUP BY ul.num_week
),
peak_week AS (
    SELECT wa.week_number
    FROM week_activity wa
    ORDER BY wa.avg_s_all_total DESC
    LIMIT 1
)
SELECT
    ul.name_osno,
    pw.week_number,
    ROUND(AVG(ul.s_all)::numeric, 2) AS avg_s_all,
    ROUND(AVG(ul.s_course_viewed)::numeric, 2) AS avg_s_course_viewed,
    ROUND(AVG(ul.s_q_attempt_viewed)::numeric, 2) AS avg_s_q_attempt_viewed
FROM user_logs ul
JOIN peak_week pw ON pw.week_number = ul.num_week
GROUP BY ul.name_osno, pw.week_number
ORDER BY ul.name_osno;
               """)
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 24, Decimal('20.79'), Decimal('3.78'), Decimal('4.94'))
(2, 24, Decimal('28.69'), Decimal('4.73'), Decimal('5.42'))
